# Editorial Scraper for the Ongoing Iran-US War

This notebook stays separate from your news-article notebook and writes to the same separate folder: `output_public_opinion`.

Because your institute firewall blocks Reddit, YouTube, and X/Twitter, this version uses a firewall-friendly method:
- RSS feeds where official opinion feeds are available
- open opinion/editorial listing pages where RSS is not practical
- full-text extraction with `newspaper3k`
- strict war-context filtering so only Iran-US-war-related viewpoints are kept

So instead of failing on blocked social platforms, this notebook collects public-facing viewpoints such as opinions, editorials, op-eds, columns, and letters to the editor from multiple outlets and perspectives.

The output of this scraper file is taken to Datasets/News_Opinion as `News_Public_Opinion.csv`


>For the benefit of the scraper to work properly, please do not change any part of this scraper file

In [1]:
# Cell 1 - Install dependencies
%pip install -q pandas requests feedparser newspaper3k lxml_html_clean beautifulsoup4


Note: you may need to restart the kernel to use updated packages.


In [2]:
# Cell 2 - Imports and folders
import html
import json
import re
import time
from datetime import datetime, timedelta, timezone
from email.utils import parsedate_to_datetime
from pathlib import Path
from urllib.parse import urljoin, urlsplit, urlunsplit

import feedparser
import pandas as pd
import requests
from bs4 import BeautifulSoup
from IPython.display import display
from newspaper import Article
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# 1. Define the Output Directory
OUTPUT_DIR = Path(r'D:\COLLEGE\MBA\SEMESTERS\IV Semester\MBAG - 408 - TWA\24030302038_Master_Project\Datasets\Scraped Datasets\Editorial Opinions')
OUTPUT_DIR.mkdir(exist_ok=True)

# 2. Check if the files already exist using a pattern search
existing_files = list(OUTPUT_DIR.glob("iran_us_public_opinion_*.csv"))
ALREADY_SCRAPED = len(existing_files) > 0

if ALREADY_SCRAPED:
    print(f"✅ Already scraped. Found existing dataset: '{existing_files[0].name}'.")
    print(f"Delete the files in '{OUTPUT_DIR}' if you want to scrape again.")
else:
    print("No existing dataset found. Ready to scrape.")


✅ Already scraped. Found existing dataset: 'iran_us_public_opinion_20260324_113727.csv'.
Delete the files in 'D:\COLLEGE\MBA\SEMESTERS\IV Semester\MBAG - 408 - TWA\24030302038_Master_Project\Datasets\Scraped Datasets\Editorial Opinions' if you want to scrape again.


In [3]:
# Cell 3 - Research settings
TOPIC_LABEL = "Public opinion / editorial framing on the ongoing Iran-US war"
LOOKBACK_DAYS = 120
REQUEST_DELAY_SECONDS = 1.0
MIN_FULL_TEXT_CHARS = 250
MAX_ITEMS_PER_LISTING = None
FIREWALL_FRIENDLY_MODE = True

OPINION_SOURCES = [
    {
        "source": "The Guardian",
        "perspective": "UK / Western liberal",
        "country_or_region": "United Kingdom",
        "collection_method": "html_listing",
        "opinion_type": "Comment / Opinion",
        "listing_urls": [
            {
                "listing_name": "Comment is Free",
                "listing_url": "https://www.theguardian.com/commentisfree",
            }
        ],
        "allowed_url_hints": ["/commentisfree/"],
        "blocked_url_hints": ["/all", "/audio", "/gallery", "/video", "/cartoon"],
    },
    {
        "source": "Al Jazeera",
        "perspective": "Qatar / Arab regional",
        "country_or_region": "Qatar",
        "collection_method": "html_listing",
        "opinion_type": "Opinion",
        "listing_urls": [
            {
                "listing_name": "Opinion",
                "listing_url": "https://www.aljazeera.com/opinion/",
            }
        ],
        "allowed_url_hints": ["/opinion/", "/opinions/"],
        "blocked_url_hints": ["/liveblog", "/podcasts", "/video"],
    },
    {
        "source": "Arab News",
        "perspective": "Saudi / Arab regional",
        "country_or_region": "Saudi Arabia",
        "collection_method": "html_listing",
        "opinion_type": "Opinion / Letters",
        "listing_urls": [
            {
                "listing_name": "Opinion",
                "listing_url": "https://www.arabnews.com/opinion",
            },
            {
                "listing_name": "Letters to the Editor",
                "listing_url": "https://www.arabnews.com/category/main-category/opinion/letters",
            },
        ],
        "allowed_url_hints": ["/node/", "/opinion", "/letters"],
        "blocked_url_hints": ["/jobs", "/rss", "/contact"],
    },
    {
        "source": "Jerusalem Post",
        "perspective": "Israeli",
        "country_or_region": "Israel",
        "collection_method": "html_listing",
        "opinion_type": "Opinion / Editorial",
        "listing_urls": [
            {
                "listing_name": "Opinion",
                "listing_url": "https://www.jpost.com/opinion",
            }
        ],
        "allowed_url_hints": ["/opinion"],
        "blocked_url_hints": ["/rss", "/tags/", "/conference", "/premium"],
    },
    {
        "source": "Indian Express",
        "perspective": "Indian mainstream",
        "country_or_region": "India",
        "collection_method": "rss",
        "opinion_type": "Opinion / Editorial",
        "feed_urls": [
            {
                "listing_name": "Opinion",
                "feed_url": "https://indianexpress.com/section/opinion/feed/",
            },
            {
                "listing_name": "Editorials",
                "feed_url": "https://indianexpress.com/section/opinion/editorials/feed/",
            },
        ],
        "allowed_url_hints": ["/opinion/"],
        "blocked_url_hints": ["/videos/", "/photos/"],
    },
    {
        "source": "Firstpost",
        "perspective": "Indian digital",
        "country_or_region": "India",
        "collection_method": "rss",
        "opinion_type": "Opinion",
        "feed_urls": [
            {
                "listing_name": "Opinion",
                "feed_url": "https://www.firstpost.com/commonfeeds/v1/mfp/rss/opinion.xml",
            }
        ],
        "allowed_url_hints": ["/opinion/"],
        "blocked_url_hints": ["/videos/", "/photos/"],
    },
    {
        "source": "Daily Sabah",
        "perspective": "Turkish / regional",
        "country_or_region": "Turkey",
        "collection_method": "rss",
        "opinion_type": "Opinion / Editorial / Reader letters",
        "feed_urls": [
            {
                "listing_name": "Columns",
                "feed_url": "https://www.dailysabah.com/rss/opinion/columns",
            },
            {
                "listing_name": "Op-Ed",
                "feed_url": "https://www.dailysabah.com/rss/opinion/op-ed",
            },
            {
                "listing_name": "Reader's Corner",
                "feed_url": "https://www.dailysabah.com/rss/opinion/readers-corner",
            },
            {
                "listing_name": "Editorial",
                "feed_url": "https://www.dailysabah.com/rss/opinion/editorial",
            },
        ],
        "allowed_url_hints": ["/opinion/"],
        "blocked_url_hints": ["/gallery", "/video"],
    },
]

WAR_TERM_GROUPS = {
    "iran": {
        "iran": r"\biran\b",
        "iranian": r"\biranian\b",
        "tehran": r"\btehran\b",
        "irgc": r"\birgc\b",
        "khamenei": r"\bkhamenei\b",
        "hormuz": r"\bhormuz\b",
        "persian gulf": r"\bpersian gulf\b",
    },
    "us": {
        "united states": r"\bunited states\b",
        "usa": r"\bu\.s\.a\b|\busa\b",
        "america": r"\bamerica\b|\bamerican\b",
        "washington": r"\bwashington\b",
        "white house": r"\bwhite house\b",
        "pentagon": r"\bpentagon\b",
        "centcom": r"\bcentcom\b",
        "trump": r"\btrump\b",
        "biden": r"\bbiden\b",
    },
    "war": {
        "war": r"\bwar\b",
        "conflict": r"\bconflict\b",
        "strike": r"\bstrike\b|\bstrikes\b",
        "attack": r"\battack\b|\battacks\b",
        "missile": r"\bmissile\b|\bmissiles\b",
        "drone": r"\bdrone\b|\bdrones\b",
        "retaliation": r"\bretaliat(?:e|ion)\b",
        "ceasefire": r"\bceasefire\b",
        "bombing": r"\bbombing\b|\bbombard(?:ed|ment)?\b",
        "military": r"\bmilitary\b",
    },
    "regional": {
        "israel": r"\bisrael\b|\bisraeli\b",
        "gaza": r"\bgaza\b",
        "hezbollah": r"\bhezbollah\b",
        "houthis": r"\bhouthis?\b",
        "middle east": r"\bmiddle east\b",
        "gulf": r"\bgulf\b",
        "iraq": r"\biraq\b",
        "syria": r"\bsyria\b",
        "lebanon": r"\blebanon\b",
        "saudi": r"\bsaudi\b",
        "uae": r"\buae\b|\bunited arab emirates\b",
    },
}

COMPILED_WAR_PATTERNS = {
    group: [(label, re.compile(pattern, re.IGNORECASE)) for label, pattern in patterns.items()]
    for group, patterns in WAR_TERM_GROUPS.items()
}

print("Firewall-friendly mode:", FIREWALL_FRIENDLY_MODE)
print("Sources configured     :", len(OPINION_SOURCES))
print("Lookback days          :", LOOKBACK_DAYS)
print("Output folder          :", OUTPUT_DIR.resolve())


Firewall-friendly mode: True
Sources configured     : 7
Lookback days          : 120
Output folder          : D:\COLLEGE\MBA\SEMESTERS\IV Semester\MBAG - 408 - TWA\24030302038_Master_Project\Datasets\Scraped Datasets\Editorial Opinions


In [4]:
# Cell 4 - Review the source catalog
source_catalog_rows = []

for source in OPINION_SOURCES:
    listing_configs = source.get("feed_urls") or source.get("listing_urls") or []
    for config in listing_configs:
        source_catalog_rows.append(
            {
                "source": source["source"],
                "perspective": source["perspective"],
                "country_or_region": source["country_or_region"],
                "opinion_type": source["opinion_type"],
                "collection_method": source["collection_method"],
                "listing_name": config.get("listing_name", ""),
                "url": config.get("feed_url", config.get("listing_url", "")),
            }
        )

source_catalog_df = pd.DataFrame(source_catalog_rows)
display(source_catalog_df)

,source,perspective,country_or_region,opinion_type,collection_method,listing_name,url
0,The Guardian,UK / Western liberal,United Kingdom,Comment / Opinion,html_listing,Comment is Free,https://www.theguardian.com/commentisfree
1,Al Jazeera,Qatar / Arab regional,Qatar,Opinion,html_listing,Opinion,https://www.aljazeera.com/opinion/
2,Arab News,Saudi / Arab regional,Saudi Arabia,Opinion / Letters,html_listing,Opinion,https://www.arabnews.com/opinion
3,Arab News,Saudi / Arab regional,Saudi Arabia,Opinion / Letters,html_listing,Letters to the Editor,https://www.arabnews.com/category/main-categor...
4,Jerusalem Post,Israeli,Israel,Opinion / Editorial,html_listing,Opinion,https://www.jpost.com/opinion
5,Indian Express,Indian mainstream,India,Opinion / Editorial,rss,Opinion,https://indianexpress.com/section/opinion/feed/
6,Indian Express,Indian mainstream,India,Opinion / Editorial,rss,Editorials,https://indianexpress.com/section/opinion/edit...
7,Firstpost,Indian digital,India,Opinion,rss,Opinion,https://www.firstpost.com/commonfeeds/v1/mfp/r...
8,Daily Sabah,Turkish / regional,Turkey,Opinion / Editorial / Reader letters,rss,Columns,https://www.dailysabah.com/rss/opinion/columns
9,Daily Sabah,Turkish / regional,Turkey,Opinion / Editorial / Reader letters,rss,Op-Ed,https://www.dailysabah.com/rss/opinion/op-ed


In [5]:
# Cell 5 - Helpers and opinion scraping functions
DEFAULT_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36",
    "Accept": "application/rss+xml, application/xml, text/xml, application/atom+xml, text/html;q=0.9, */*;q=0.8",
}


def build_http_session():
    session = requests.Session()
    retry_strategy = Retry(
        total=2,
        connect=2,
        read=2,
        backoff_factor=0.5,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=frozenset(["GET", "HEAD"]),
    )
    adapter = HTTPAdapter(max_retries=retry_strategy)
    session.mount("http://", adapter)
    session.mount("https://", adapter)
    session.headers.update(DEFAULT_HEADERS)
    return session


HTTP_SESSION = build_http_session()


def clean_text(text):
    text = html.unescape(text or "")
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def normalize_url(url):
    parsed = urlsplit(url or "")
    path = parsed.path.rstrip("/")
    return urlunsplit((parsed.scheme.lower(), parsed.netloc.lower(), path, "", ""))


def title_key(title):
    value = re.sub(r"[^a-z0-9]+", " ", (title or "").lower())
    return re.sub(r"\s+", " ", value).strip()


def matches_any(text, hints):
    lowered = (text or "").lower()
    return any(hint.lower() in lowered for hint in (hints or []))


def same_site(candidate_url, listing_url):
    candidate_domain = urlsplit(candidate_url).netloc.lower().replace("www.", "")
    listing_domain = urlsplit(listing_url).netloc.lower().replace("www.", "")
    return candidate_domain.endswith(listing_domain) or listing_domain.endswith(candidate_domain)


def parse_entry_date(entry):
    for attr in ["published_parsed", "updated_parsed", "created_parsed"]:
        value = getattr(entry, attr, None) or entry.get(attr)
        if value:
            try:
                return datetime(*value[:6], tzinfo=timezone.utc)
            except Exception:
                pass

    for attr in ["published", "updated", "created"]:
        value = getattr(entry, attr, None) or entry.get(attr)
        if value:
            try:
                parsed = parsedate_to_datetime(value)
                if parsed.tzinfo is None:
                    parsed = parsed.replace(tzinfo=timezone.utc)
                return parsed.astimezone(timezone.utc)
            except Exception:
                pass

    return None


def safe_datetime_to_iso(value):
    if not value:
        return ""
    if value.tzinfo is None:
        value = value.replace(tzinfo=timezone.utc)
    return value.astimezone(timezone.utc).isoformat().replace("+00:00", "Z")


def iso_to_datetime(value):
    if not value:
        return None
    try:
        return datetime.fromisoformat(value.replace("Z", "+00:00")).astimezone(timezone.utc)
    except Exception:
        return None


def feed_url_candidates(feed_url):
    candidates = [feed_url]
    if feed_url.startswith("http://"):
        candidates.insert(0, "https://" + feed_url[len("http://"):])
    return list(dict.fromkeys(candidates))


def fetch_feed(feed_url, timeout=25):
    errors = []
    last_feed = None

    for candidate in feed_url_candidates(feed_url):
        try:
            response = HTTP_SESSION.get(candidate, timeout=timeout, allow_redirects=True)
            response.raise_for_status()
            parsed = feedparser.parse(response.content)
            if parsed.entries:
                return parsed, candidate, None
            content_type = response.headers.get("content-type", "unknown")
            errors.append(f"{candidate} returned no RSS entries ({content_type})")
            last_feed = parsed
        except Exception as exc:
            errors.append(f"{candidate} -> {exc}")

    try:
        parsed = feedparser.parse(feed_url)
        if parsed.entries:
            return parsed, feed_url, None
        last_feed = parsed
        if getattr(parsed, "bozo", 0):
            errors.append(str(getattr(parsed, "bozo_exception", "unknown feed issue")))
    except Exception as exc:
        errors.append(f"{feed_url} -> {exc}")

    if last_feed is None:
        last_feed = feedparser.FeedParserDict(entries=[])

    return last_feed, feed_url, " | ".join(errors[:3])


def fetch_html(url, timeout=25):
    response = HTTP_SESSION.get(url, timeout=timeout, allow_redirects=True)
    response.raise_for_status()
    return response.text


def extract_listing_links(listing_url, source_config):
    html_text = fetch_html(listing_url)
    soup = BeautifulSoup(html_text, "html.parser")
    links = []
    seen = set()
    allowed_hints = source_config.get("allowed_url_hints", [])
    blocked_hints = source_config.get("blocked_url_hints", [])
    normalized_listing_url = normalize_url(listing_url)

    for tag in soup.find_all("a", href=True):
        href = urljoin(listing_url, tag.get("href", "").strip())
        normalized = normalize_url(href)
        if not normalized or normalized == normalized_listing_url or normalized in seen:
            continue
        if not same_site(normalized, listing_url):
            continue
        if matches_any(normalized, blocked_hints):
            continue
        if allowed_hints and not matches_any(normalized, allowed_hints):
            continue

        title_hint = clean_text(tag.get_text(" ", strip=True))
        if len(title_hint) < 4:
            title_hint = ""

        seen.add(normalized)
        links.append({"url": normalized, "title_hint": title_hint})

    if MAX_ITEMS_PER_LISTING is not None:
        return links[:MAX_ITEMS_PER_LISTING]
    return links


def detect_war_context(text):
    cleaned = clean_text(text)
    matched = {}

    for group, patterns in COMPILED_WAR_PATTERNS.items():
        hits = sorted({label for label, pattern in patterns if pattern.search(cleaned)})
        if hits:
            matched[group] = hits

    score = sum(len(hits) for hits in matched.values())
    return matched, score


def article_is_relevant(title, summary="", full_text=""):
    preview_text = clean_text(f"{title}. {summary}")
    preview_matches, preview_score = detect_war_context(preview_text)
    full_matches, full_score = detect_war_context(full_text)

    def passes(match_map, score):
        has_iran = "iran" in match_map
        has_us = "us" in match_map
        has_war = "war" in match_map
        has_regional = "regional" in match_map
        return has_iran and (has_us or has_war or (has_regional and score >= 2) or score >= 4)

    keep = passes(preview_matches, preview_score) or passes(full_matches, full_score)
    best_matches = full_matches if full_score >= preview_score else preview_matches
    best_score = max(preview_score, full_score)
    matched_terms = sorted({term for terms in best_matches.values() for term in terms})
    return keep, best_score, ", ".join(sorted(best_matches)), ", ".join(matched_terms)


def extract_article(url):
    try:
        article = Article(
            url=url,
            language="en",
            browser_user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
        )
        article.download()
        article.parse()
        full_text = clean_text(article.text)
        if len(full_text) < MIN_FULL_TEXT_CHARS:
            full_text = ""
        authors = ", ".join([clean_text(author) for author in article.authors if clean_text(author)])
        return {
            "title": clean_text(article.title),
            "summary": clean_text(article.meta_description),
            "full_text": full_text,
            "authors": authors,
            "published_at": safe_datetime_to_iso(article.publish_date),
        }
    except Exception:
        return {
            "title": "",
            "summary": "",
            "full_text": "",
            "authors": "",
            "published_at": "",
        }
    
OPINION_COLUMNS = [
    "topic",
    "source",
    "perspective",
    "country_or_region",
    "opinion_type",
    "listing_name",
    "listing_url",
    "published_at",
    "title",
    "summary",
    "full_text",
    "url",
    "domain",
    "author",
    "relevance_score",
    "matched_groups",
    "matched_terms",
    "used_fallback_text",
    "full_text_chars",
    "collection_method",
]


def build_record(source, listing_name, listing_url, url, title, summary, full_text, author, published_at, score, matched_groups, matched_terms, collection_method):
    final_text = full_text or clean_text(f"{title}. {summary}")
    return {
        "topic": TOPIC_LABEL,
        "source": source["source"],
        "perspective": source["perspective"],
        "country_or_region": source["country_or_region"],
        "opinion_type": source["opinion_type"],
        "listing_name": listing_name,
        "listing_url": listing_url,
        "published_at": published_at,
        "title": title,
        "summary": summary,
        "full_text": final_text,
        "url": url,
        "domain": urlsplit(url).netloc.lower(),
        "author": author,
        "relevance_score": score,
        "matched_groups": matched_groups,
        "matched_terms": matched_terms,
        "used_fallback_text": not bool(full_text),
        "full_text_chars": len(final_text),
        "collection_method": collection_method,
    }


def collect_from_rss_source(source, cutoff):
    records = []
    seen_urls = set()
    seen_titles = set()

    for feed_config in source.get("feed_urls", []):
        listing_name = feed_config["listing_name"]
        feed_url = feed_config["feed_url"]
        print("=" * 90)
        print(f"Source:   {source['source']}")
        print(f"Feed:     {listing_name}")
        print(f"Feed URL: {feed_url}")

        feed, resolved_feed_url, warning = fetch_feed(feed_url)
        if resolved_feed_url != feed_url:
            print(f"Resolved: {resolved_feed_url}")
        if warning:
            print("Warning :", warning)

        entries = list(getattr(feed, "entries", []) or [])
        if MAX_ITEMS_PER_LISTING is not None:
            entries = entries[:MAX_ITEMS_PER_LISTING]

        kept = 0

        for entry in entries:
            published_dt = parse_entry_date(entry)
            if published_dt and published_dt < cutoff:
                continue

            url = getattr(entry, "link", "") or entry.get("id", "")
            title = clean_text(getattr(entry, "title", "") or entry.get("title", ""))
            summary = clean_text(
                getattr(entry, "summary", "")
                or entry.get("summary", "")
                or getattr(entry, "description", "")
                or entry.get("description", "")
            )
            normalized_url = normalize_url(url)
            title_token = (source["source"], title_key(title))

            if not url or not title:
                continue
            if matches_any(url, source.get("blocked_url_hints", [])):
                continue
            if source.get("allowed_url_hints") and not matches_any(url, source.get("allowed_url_hints", [])):
                continue
            if normalized_url in seen_urls or title_token in seen_titles:
                continue

            article_meta = extract_article(url)
            final_title = article_meta["title"] or title
            final_summary = article_meta["summary"] or summary
            full_text = article_meta["full_text"]
            published_at = article_meta["published_at"] or safe_datetime_to_iso(published_dt)

            keep_item, score, matched_groups, matched_terms = article_is_relevant(final_title, final_summary, full_text or summary)
            if not keep_item:
                continue

            records.append(
                build_record(
                    source=source,
                    listing_name=listing_name,
                    listing_url=resolved_feed_url,
                    url=url,
                    title=final_title,
                    summary=final_summary,
                    full_text=full_text,
                    author=article_meta["authors"],
                    published_at=published_at,
                    score=score,
                    matched_groups=matched_groups,
                    matched_terms=matched_terms,
                    collection_method="rss + newspaper3k",
                )
            )
            seen_urls.add(normalized_url)
            seen_titles.add(title_token)
            kept += 1

            if REQUEST_DELAY_SECONDS > 0:
                time.sleep(REQUEST_DELAY_SECONDS)

        print(f"Collected: {kept} relevant opinion texts")

    return records


def collect_from_listing_source(source, cutoff):
    records = []
    seen_urls = set()
    seen_titles = set()

    for listing_config in source.get("listing_urls", []):
        listing_name = listing_config["listing_name"]
        listing_url = listing_config["listing_url"]
        print("=" * 90)
        print(f"Source:      {source['source']}")
        print(f"Listing:     {listing_name}")
        print(f"Listing URL: {listing_url}")

        try:
            candidates = extract_listing_links(listing_url, source)
            print(f"Candidates:  {len(candidates)} links")
        except Exception as exc:
            print(f"Listing fetch failed: {exc}")
            continue

        kept = 0

        for candidate in candidates:
            url = candidate["url"]
            normalized_url = normalize_url(url)
            if normalized_url in seen_urls:
                continue

            article_meta = extract_article(url)
            final_title = article_meta["title"] or candidate["title_hint"]
            final_summary = article_meta["summary"]
            full_text = article_meta["full_text"]
            published_at = article_meta["published_at"]
            published_dt = iso_to_datetime(published_at)

            if published_dt and published_dt < cutoff:
                continue
            if not final_title:
                continue

            title_token = (source["source"], title_key(final_title))
            if title_token in seen_titles:
                continue

            keep_item, score, matched_groups, matched_terms = article_is_relevant(final_title, final_summary, full_text)
            if not keep_item:
                continue

            records.append(
                build_record(
                    source=source,
                    listing_name=listing_name,
                    listing_url=listing_url,
                    url=url,
                    title=final_title,
                    summary=final_summary,
                    full_text=full_text,
                    author=article_meta["authors"],
                    published_at=published_at,
                    score=score,
                    matched_groups=matched_groups,
                    matched_terms=matched_terms,
                    collection_method="html listing + newspaper3k",
                )
            )
            seen_urls.add(normalized_url)
            seen_titles.add(title_token)
            kept += 1

            if REQUEST_DELAY_SECONDS > 0:
                time.sleep(REQUEST_DELAY_SECONDS)

        print(f"Collected:   {kept} relevant opinion texts")

    return records


def collect_public_opinion():
    cutoff = datetime.now(timezone.utc) - timedelta(days=LOOKBACK_DAYS)
    all_records = []

    for source in OPINION_SOURCES:
        if source["collection_method"] == "rss":
            all_records.extend(collect_from_rss_source(source, cutoff))
        else:
            all_records.extend(collect_from_listing_source(source, cutoff))

    if not all_records:
        return pd.DataFrame(columns=OPINION_COLUMNS)

    df = pd.DataFrame(all_records)
    df = df[OPINION_COLUMNS].copy()
    df["published_at"] = df["published_at"].fillna("").astype(str)
    df = df.drop_duplicates(subset=["source", "url", "title", "full_text"]).reset_index(drop=True)
    df = df.sort_values(["published_at", "source", "listing_name"], ascending=[False, True, True], na_position="last").reset_index(drop=True)
    return df


In [6]:
# Cell 6 - Main collection runner
if ALREADY_SCRAPED:
    print("Skipping collection... Dataset already exists in the output folder.")
else:
    public_opinion_df = collect_public_opinion()

Skipping collection... Dataset already exists in the output folder.


In [7]:
# Cell 7 - Export the dataset
if ALREADY_SCRAPED:
    print("Skipping export... Dataset already exists.")
else:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    csv_path = OUTPUT_DIR / f"iran_us_public_opinion_{timestamp}.csv"
    json_path = OUTPUT_DIR / f"iran_us_public_opinion_{timestamp}.json"
    source_path = OUTPUT_DIR / f"iran_us_public_opinion_sources_{timestamp}.json"

    public_opinion_df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    public_opinion_df.to_json(json_path, orient="records", indent=2, force_ascii=False)
    source_path.write_text(json.dumps(OPINION_SOURCES, indent=2), encoding="utf-8")

    print("CSV saved to   :", csv_path.resolve())
    print("JSON saved to  :", json_path.resolve())
    print("Sources saved to:", source_path.resolve())

Skipping export... Dataset already exists.
